# Test of Allozymes UPO data

In [ ]:
from pathlib import Path

import numpy as np
import torch
import typing as T
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from copy import deepcopy
from polyleven import levenshtein
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from vsd.surrogates import CNNClassProbability, fit_cpe
from vsd.thresholds import QuantileThreshold
from vsd.proposals import DTransformerProposal, LSTMProposal, fit_ml
from vsd.acquisition import (LogPIClassiferAcquisition,
                             VariationalSearchAcquisition)
from vsd.generation import generate_candidates_reinforce

from utils import seq2intarr, to_probability, intarr2seq
from vis import CPplot

## Settings

In [ ]:
# File paths
DATAPATH = Path("../data/CSIRO_UPO_Data_package_2024.xlsx")
VALIDATIONPATH = Path("../data/AbrUPO validation.xlsx")

# Sheet properties
PB1_SHEET = {
   "name": "PB EF bin 1",
   "sequence": "Variant",
   "target": "Enrichment_fold"
}

PB2_SHEET = {
   "name": "PB EF bin 2",
   "sequence": "Variant",
   "target": "Enrichment_fold"
}

PB3_SHEET = {
   "name": "PB EF bin 3",
   "sequence": "Variant",
   "target": "Enrichment_fold"
}

S_SHEET = {
    "name" : "Original read counts",
    "mutations": "Variant",
    "target": "Read counts",
    "header": 1
}

VALIDATION_SHEET = {
    "name": "Sheet1",
    "nrows": 21
}

WTSEQ = (
    "MKTTTLLFLVGALTQTHAFPQQGVPHPLPWSPPGPNDVRAPCPMLNTLANHGFLPHNGKNITQQHTINALYNAL"
    "GIDAELATYLHQEAVTTNPVPNATTFSLNDLSRHDILEHDASLSRQDAFFGDNHDFNQTIFNQTRSYWTSPIID"
    "VKQAALARQARVNTSMATNPNYTMSELGDAFSYGETAAYIIVLGDKEAGLVNRSRVEYLFENERLPVELGWSRA"
    "RENITFDDLSTMLNKIINATGGESEFERELAKRGGVHVGWR"
)
MUT_SITES = np.array([8, 28, 65, 84, 129, 177, 179, 182, 186, 231]) - 1

# Processing options
MIN_SD_READ_COUNTS = 2
USE_ALL_S_FOR_LABELS = True  # otherwise only values that have PB AND SD

# Random State
SEED = 139
torch.manual_seed(SEED)

# CPE settings
CPE_PATH = "cpe.pt"
QUANTILE_THRESH = 0.01 if not USE_ALL_S_FOR_LABELS else 0.1  # Label threshold
KFOLDS = 5
DEVICE = "mps"
CPE_PARAMS = dict(
    ckernel=7,
    xkernel=4,
    xstride=4,
    cfilter_size=32,
    linear_size=256,
    embedding_dim=16,
    dropoutp=0.1,
    pos_encoding=True
)
CPE_OPTIM = {"lr": 1e-3, "weight_decay": 0}
CPE_STOP = {"maxiter": 200000, "n_window": 50000}
BATCH_SIZE = 64


# Generative model settings
PRIOR_PATH_SUFFIX = "prior"
GEN_CLASS = LSTMProposal
GEN_PARAMS = dict(
   num_layers=2, hidden_size=128, clip_gradients=1.0
)
# Uncomment this to train the transformer
# GEN_CLASS = DTransformerProposal
# GEN_PARAMS = dict(
#      nhead=4, num_layers=2, dim_feedforward=128, clip_gradients=1.0
# )
GEN_OPTIM =  {"lr": 5e-4, "weight_decay": 1e-4}
GEN_STOP = {"maxiter": 20000, "n_window": 2000}
PVAL = 0.01  # Proportion of data to use for validating sequence generator

# Fine-tuned model settings
FINETUNE_PATH_SUFFIX = "finetuned"

# VSD Settings
VARD_PATH_SUFFIX = "vard"
VSD_OPTIM = dict(lr=1e-4, weight_decay=0.0)
VSD_STOP = dict(maxiter=20000, n_window=3000)
VSD_SAMPLES = 512
VSD_BATCH_SIZE = 128

# Output directory -- follows the generative model being trained, so a
# transformer run cannot overwrite the LSTM results.
RESPATH = Path("../results") / ("lstm" if GEN_CLASS is LSTMProposal else "tfm")
RESPATH.mkdir(parents=True, exist_ok=True)

## Useful functions

In [ ]:
# Turn mutations into a sequence
def muts2seq(muts: str, wt: str) -> str:
    if muts == "WT" or pd.isna(muts):
        return wt
    muts = muts.split("_")
    seq = wt
    for m in muts:
        o, ind, n = m[0], int(m[1:-1]) - 1, m[-1]
        if o != seq[ind]:
            raise RuntimeError(f"Substitution {o} @ {ind+1} does not match WT!")
        seq = seq[:ind] + n + seq[ind+1:]
    return seq

# Rank -- bigger is better
def rank(arr: pd.Series, higher_better: bool=True) -> np.ndarray:
    arr = -arr if higher_better else arr
    ranks = rankdata(arr, method="average", nan_policy="omit")
    ranks[np.isnan(ranks)] = np.nanmax(ranks) + 1
    return ranks


def rankagg(ranks: pd.DataFrame) -> pd.Series | np.ndarray:
    """Implements rank aggregation as described in Bedo, Ong 2016.

    Bedő, J., & Ong, C. S. (2016). Multivariate Spearman's ρ for Aggregating
    Ranks Using Copulas. Journal of Machine Learning Research, 17(201), 1-30.
    """
    # normalise to (0, 1)
    ranks = ranks - ranks.min(axis=0) + 1
    ranks = ranks / (ranks.max(axis=0) + 1)

    # Geometric mean ranking
    aggrank = np.exp(np.log(ranks).mean(axis=1))
    #aggrank = rank(aggrank, higher_better=False)
    #aggrank /= aggrank.max() + 1
    return aggrank


# Plot optimization trace
def plot_trace(
    its: T.Sequence,
    losses: T.Sequence,
    validation: T.Optional[T.Sequence]=None,
    title: str="",
    savepath: T.Optional[Path | str]=None
):
    plt.plot(its, losses, label="training loss")
    if validation is not None:
        plt.plot(its, validation, label="validation loss")
    plt.title(title)
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    if savepath is not None:
        plt.savefig(savepath, dpi=200)


def wt_diversity(variants: T.Sequence) -> float:
    """Average diversity (edit distance) from wildtype"""
    div = 0.
    for s in variants:
        div += levenshtein(WTSEQ, s)
    div /= len(variants)
    return div

## Load data and join

In [ ]:
# Propylbenzene data
df_pb = [
    pd.read_excel(
        DATAPATH,
        sheet_name=pb["name"],
        usecols=[pb["sequence"], pb["target"]]
    )
    for pb in [PB1_SHEET, PB2_SHEET, PB3_SHEET]
]

# Styrene data
df_sd = pd.read_excel(
        DATAPATH,
        sheet_name=S_SHEET["name"],
        header=S_SHEET["header"],
        usecols=[S_SHEET["mutations"], S_SHEET["target"]]
    )

In [ ]:
# Join PB Bin data
for i, df in enumerate(df_pb):
    df.rename(columns={"Enrichment_fold": f"EF_B{i+1}"}, inplace=True)
    df.set_index("Variant", inplace=True)

df_pb_join = df_pb[0].join(df_pb[1], how="outer")
df_pb_join = df_pb_join.join(df_pb[2], how="outer")
print(f"Length of PB EF data = {len(df_pb_join)}")

In [ ]:
# Filter and join Styrene data
df_sd = df_sd.dropna()
df_sd["Variant"] = df_sd["Variant"].map(lambda s: muts2seq(s, WTSEQ))
df_sd.set_index("Variant", inplace=True)
df_sd = df_sd[df_sd["Read counts"] >= MIN_SD_READ_COUNTS]
print(f"Length of S read count data = {len(df_sd)}")

In [ ]:
# Join it all
df = df_pb_join.join(df_sd, how="right" if USE_ALL_S_FOR_LABELS else "left")
print(f"Length of all joined data = {len(df)}")

In [ ]:
# Get all sequences that we have seen minimum SD read counts
Sall = set(df_sd.index).union(df.index)
print(f"Length of valid SD sequences {len(Sall)}")

## Make rank columns and combine

In [ ]:
# Rank each column
cols = list(df.columns)
rcols = [f"{col}_rank" for col in cols]
for col, rcol in zip(cols, rcols):
    df[rcol] = rank(df[col])

# Geometric rank aggregation
df["rank"] = rankagg(df[rcols])

In [ ]:
df.to_csv(RESPATH / "ranked_variants.csv", encoding="utf-8-sig")

# Explore potential targets

In [ ]:
# Marginal historgrams
df["log10 read counts"] = np.log10(df["Read counts"])
# fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
# df[["EF_B1", "EF_B2", "EF_B3", "log10 read counts"]].hist(bins=50, ax=ax)
fig, axs = plt.subplots(2, 2, figsize=(8, 8), dpi=150)
xlables = ["Enrichment factor"] * 3 + ["Reads"]
for c, ax, xl in zip(
    ["EF_B1", "EF_B2", "EF_B3", "log10 read counts"],
    axs.flatten(),
    xlables
):
    df[[c]].plot.kde(ax=ax, color="green")
    ax.set_xlabel(xl)
    ax.set_ylabel("Density")
    ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Original
scatter_kws = {'alpha': 0.5,  'marker': "."}
sns.pairplot(df[cols + ["rank"]], hue="rank", palette="magma", plot_kws=scatter_kws)
plt.show()

In [ ]:
# Ranks
scatter_kws = {'alpha': 0.5,  'marker': "."}
sns.pairplot(df[rcols + ["rank"]], hue="rank", palette="magma", plot_kws=scatter_kws)
plt.show()

## Load validation set

And compare to ranks

In [ ]:
df_val = pd.read_excel(VALIDATIONPATH, sheet_name=VALIDATION_SHEET["name"],
                       nrows=VALIDATION_SHEET["nrows"])
df_val["Variants"] = df_val["Mutations"].map(lambda s: muts2seq(s, wt=WTSEQ))

In [ ]:
indf = df_val["Variants"].isin(df.index)
df.loc[df_val[indf]["Variants"]]

In [ ]:
corr_cols = [
    "EF_B1",
    "EF_B2",
    "EF_B3",
    "Read counts",
    "rank"
]
for c in corr_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
corr = df[corr_cols].corr(method="spearman")
corr.style.background_gradient(cmap='coolwarm').format(precision=2)

In [ ]:
corr.to_csv(RESPATH / "targets_spearman_corr.csv", encoding="utf-8-sig")

## Class probability estimator training

In [ ]:
# Separate targets and sequences, tokenize
y, S = torch.tensor(df["rank"].values), df.index.values
X = seq2intarr(S)

# Get quantiles and make labels
qthresh = QuantileThreshold(percentile=QUANTILE_THRESH)
rank_thresh = qthresh(y)
z = (-y >= -rank_thresh).numpy()

In [ ]:
rank_thresh

### Cross validation

In [ ]:
# Cross val
kfold = StratifiedKFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
cpeplot = CPplot()
lls, baccs = [], []

def make_cpe():
    cpe = CNNClassProbability(
        seq_len=X.shape[1],
        alpha_len=20,
        **CPE_PARAMS
    )
    return cpe

cpe_path = RESPATH / CPE_PATH
if not cpe_path.exists():
    for k, (tri, tsi) in enumerate(kfold.split(X, z)):
        Xr, yr, zr = X[tri], y[tri], z[tri]
        Xs, zs = X[tsi], z[tsi]

        cpe = make_cpe()

        fit_cpe(
            cpe,
            Xr,
            -yr,
            best_f=-rank_thresh,
            batch_size=BATCH_SIZE,
            stop_options=CPE_STOP,
            optimizer_options=CPE_OPTIM,
            device=DEVICE,
            seed=SEED
        )

        pr = to_probability(cpe(Xr.to(DEVICE)))
        ps = to_probability(cpe(Xs.to(DEVICE)))
        ll = log_loss(zs, np.vstack((1 - ps, ps)).T)
        bacc = balanced_accuracy_score(zs, ps > 0.5)
        print(f"Fold {k}: Log loss = {ll:.3f}, Bal. Acc. = {bacc:.3f}")
        lls.append(ll)
        baccs.append(bacc)
        cpeplot.update(zr, zs, pr, ps)

    print(
        f"Final: Log loss = {np.mean(lls):.3f}, "
        f"Bal. Acc = {np.mean(baccs):.3f}"
    )
    cpeplot.plot(scoring=log_loss)
    plt.savefig(RESPATH / "cpe_kfold.png")
    plt.show()

### Training

In [ ]:
cpe = make_cpe()

if not cpe_path.exists():
    its, losses = [], []
    def callback(it, loss, _):
        its.append(it)
        losses.append(loss.detach().to("cpu").numpy())

    fit_cpe(
        cpe,
        X,
        -y,
        best_f=-rank_thresh,
        batch_size=BATCH_SIZE,
        stop_options=CPE_STOP,
        optimizer_options=CPE_OPTIM,
        callback=callback,
        device=DEVICE,
        seed=SEED,
    )
    torch.save(cpe.state_dict(), cpe_path)

    plot_trace(
        its,
        losses,
        title="Final surrogate training",
        savepath=RESPATH / "cpe_trace.png"
    )
    plt.show()

else:
    sd = torch.load(
        f=cpe_path,
        map_location=torch.device(DEVICE),
        weights_only=True,
    )
    cpe.load_state_dict(sd)
    cpe = cpe.to(DEVICE)
    cpe.eval()  # fit_* leaves the model in eval mode; match that when loading

## Generative model training

Make a prior generative model on all the sequences seen so far.

In [ ]:
# Use all sequences that are "viable" to train initial model
Xall = seq2intarr(Sall)

# Make some random validation data to help with overfitting
rand = np.random.RandomState(seed=SEED)
val_mask = rand.binomial(n=1, p=PVAL, size=len(Xall)).astype(bool)
Xr = Xall[~val_mask]
Xv = Xall[val_mask]

In [ ]:
gen = GEN_CLASS(
    d_features=X.shape[1],
    k_categories=20,
    **GEN_PARAMS
)
prior_path = RESPATH / f"{PRIOR_PATH_SUFFIX}_{GEN_CLASS.__name__}.pt"

if not prior_path.exists():
    its, losses, validation = [], [], []

    def callback(it, loss, vloss):
            its.append(it)
            losses.append(loss.detach().to("cpu").numpy())
            validation.append(vloss.to("cpu").numpy())
            if it % 100 == 0:
                print(
                    f" - iter: {it}, "
                    f"train loss: {losses[-1]:.3f}, "
                    f"validation loss: {validation[-1]:.3f}"
                )

    fit_ml(
        gen,
        Xr,
        Xv,
        batch_size=BATCH_SIZE,
        optimizer_options=GEN_OPTIM,
        stop_options=GEN_STOP,
        device=DEVICE,
        callback=callback,
        seed=SEED
    )
    torch.save(gen.state_dict(), prior_path)

    plot_trace(
        its,
        losses,
        validation,
        title="Prior training",
        savepath=RESPATH / "prior_trace.png"
    )
else:
    sd = torch.load(
        f=prior_path,
        map_location=torch.device(DEVICE),
        weights_only=True,
    )
    gen.load_state_dict(sd)
    gen = gen.to(DEVICE)
    gen.eval()  # fit_* leaves the model in eval mode; match that when loading

In [ ]:
# Visualise prior mutation residues
wt_tokens = seq2intarr([WTSEQ])
Xs = torch.vstack([gen.sample(torch.Size([VSD_BATCH_SIZE])) for _ in range(1000 // VSD_BATCH_SIZE)])
diff = (Xs.to("cpu") - wt_tokens) > 0
pmutcounts = diff.sum(axis=0)
f, axs = plt.subplots(nrows=2, ncols=1, figsize=(17, 10), dpi=150)

axs[0].bar(np.arange(len(WTSEQ)), pmutcounts, label="all sites")
axs[0].bar(MUT_SITES, pmutcounts[MUT_SITES], color="r", label="targeted")
axs[0].legend()
axs[0].grid()
axs[0].set_title("Sampled sequences")

mutcounts = ((X - wt_tokens) > 0).sum(axis=0)
axs[1].bar(np.arange(len(WTSEQ)), mutcounts, label="all sites")
axs[1].legend()
axs[1].grid()
axs[1].set_title("Training sequences")

plt.show()

# Model Finetuning
As an alternative VSD refinement

In [ ]:
# Subset sequences to just the "good" ones (labelled positive)
Xgood = Xall[z]

# Make some random validation data to help with overfitting
rand = np.random.RandomState(seed=SEED)
val_mask = rand.binomial(n=1, p=PVAL, size=len(Xgood)).astype(bool)
Xgr = Xgood[~val_mask]
Xgv = Xgood[val_mask]

In [ ]:
ftmod = deepcopy(gen)
ftmod_path = RESPATH / f"{FINETUNE_PATH_SUFFIX}_{GEN_CLASS.__name__}.pt"

if not ftmod_path.exists():
    its, losses, validation = [], [], []

    def callback(it, loss, vloss):
            its.append(it)
            losses.append(loss.detach().to("cpu").numpy())
            validation.append(vloss.to("cpu").numpy())
            if it % 100 == 0:
                print(
                    f" - iter: {it}, "
                    f"train loss: {losses[-1]:.3f}, "
                    f"validation loss: {validation[-1]:.3f}"
                )

    fit_ml(
        ftmod,
        Xgr,
        Xgv,
        batch_size=BATCH_SIZE,
        optimizer_options=GEN_OPTIM,
        stop_options=GEN_STOP,
        device=DEVICE,
        callback=callback,
        seed=SEED
    )
    torch.save(ftmod.state_dict(), ftmod_path)

    plot_trace(
        its,
        losses,
        validation,
        title="Finetuning",
        savepath=RESPATH / "finetuning_trace.png"
    )
else:
    sd = torch.load(
        f=ftmod_path,
        map_location=torch.device(DEVICE),
        weights_only=True,
    )
    ftmod.load_state_dict(sd)
    ftmod = ftmod.to(DEVICE)
    ftmod.eval()  # fit_* leaves the model in eval mode; match that when loading

In [ ]:
# Visualise finetuned model mutation residues
wt_tokens = seq2intarr([WTSEQ])
Xs = torch.vstack([ftmod.sample(torch.Size([VSD_BATCH_SIZE])) for _ in range(1000 // VSD_BATCH_SIZE)])
diff = (Xs.to("cpu") - wt_tokens) > 0
pmutcounts = diff.sum(axis=0)
f, axs = plt.subplots(nrows=2, ncols=1, figsize=(17, 10), dpi=150)

axs[0].bar(np.arange(len(WTSEQ)), pmutcounts, label="all sites")
axs[0].bar(MUT_SITES, pmutcounts[MUT_SITES], color="r", label="targeted")
axs[0].legend()
axs[0].grid()
axs[0].set_title("Sampled sequences")

mutcounts = ((X - wt_tokens) > 0).sum(axis=0)
axs[1].bar(np.arange(len(WTSEQ)), mutcounts, label="all sites")
axs[1].legend()
axs[1].grid()
axs[1].set_title("Training sequences")

plt.show()

## Generative model refinement

In [ ]:
# Copy prior to variational distribution
prior = deepcopy(gen.to("cpu"))
gen = gen.to(DEVICE)
prior = prior.to(DEVICE)
prior.eval()
for p in prior.parameters():
    p.requires_grad = False


# Make the acquisition functions
piacq = LogPIClassiferAcquisition(model=cpe).to(DEVICE)
vsacq = VariationalSearchAcquisition(acquisition=piacq, prior_dist=prior)
vsacq = vsacq.to(DEVICE)

In [ ]:
# Refine or load posterior generative model
vard_path = RESPATH / f"{VARD_PATH_SUFFIX}_{GEN_CLASS.__name__}.pt"

if not vard_path.exists():

    # Callbacks and optimization
    def grad_callback(it, loss, grad, log_iters=100):
        if (it % log_iters) == 0:
            mgrad = np.mean([g.detach().to("cpu").mean() for g in grad])
            print(f"It: {it}, Loss = {loss:.3f}, Mean gradient = {mgrad:.3f}")

    Xcand, acq_cand = generate_candidates_reinforce(
        acquisition_function=vsacq,
        proposal_distribution=gen,
        optimizer_options=VSD_OPTIM,
        stop_options=VSD_STOP,
        gradient_samples=VSD_SAMPLES,
        candidate_samples=VSD_BATCH_SIZE,
        callback=grad_callback
    )

    torch.save(gen.state_dict(), vard_path)

else:
    sd = torch.load(
        f=vard_path,
        map_location=torch.device(DEVICE),
        weights_only=True,
    )
    gen.load_state_dict(sd)
    gen = gen.to(DEVICE)
    gen.eval()  # fit_* leaves the model in eval mode; match that when loading

In [ ]:
# Visualise posterior mutation residues
Xv = torch.vstack([gen.sample(torch.Size([VSD_BATCH_SIZE])) for _ in range(1000 // VSD_BATCH_SIZE)])
diff = (Xv.to("cpu") - wt_tokens) > 0
vmutcounts = diff.sum(axis=0)
f, axs = plt.subplots(nrows=3, ncols=1, figsize=(17, 15), dpi=150)

axs[0].bar(np.arange(len(WTSEQ)), pmutcounts, label="all sites")
axs[0].bar(MUT_SITES, pmutcounts[MUT_SITES], color="r", label="targeted")
axs[0].legend()
axs[0].grid()
axs[0].set_title("Prior Sampled sequences")

axs[1].bar(np.arange(len(WTSEQ)), vmutcounts, label="all sites")
axs[1].bar(MUT_SITES, vmutcounts[MUT_SITES], color="g", label="targeted")
axs[1].legend()
axs[1].grid()
axs[1].set_title("Posterior Sampled sequences")

axs[2].bar(np.arange(len(WTSEQ)), mutcounts, label="all sites")
axs[2].legend()
axs[2].grid()
axs[2].set_title("Training sequences")

plt.show()

## Rate validation sequences

Look at the likelihood of the validation sequences under the prior, likelihood and posterior models.

In [ ]:
# Get validaton data and ranks
ranks = df.loc[df_val[indf]["Variants"], "rank"]
seq_val = df_val["Variants"]
seq_val_tokens = seq2intarr(seq_val).to(DEVICE)

# Get model likelihoods and acquisition values
df_val["CPE_ll"] = cpe(seq_val_tokens).cpu().detach().numpy()
df_val["Prior_ll"] = prior.log_prob(seq_val_tokens).cpu().detach().numpy()
df_val["Finetuned_ll"] = ftmod.log_prob(seq_val_tokens).cpu().detach().numpy()
vdist_like = gen.log_prob(seq_val_tokens)
df_val["Vdist_ll"] = vdist_like.cpu().detach().numpy()
df_val["Acq"] = vsacq(seq_val_tokens, vdist_like).cpu().detach().numpy()

# Display
df_summary = df_val.set_index("Variants").drop(columns=["Variant number", "Hit concept"])
df_summary.loc[ranks.index, "Rank"] = ranks
df_summary

In [ ]:
df_summary.to_csv(RESPATH / "20_variants_scored.csv", encoding="utf-8-sig")

In [ ]:
corr_cols = [
    'PBallEF', 'PB3EF', 'PB1EF', 'PB2EF', 'PB3(B/D)', 'PB1(B/D)', 'PB2(B/D)',
    'SD1 Read count', 'SD1 percentage', 'Protein concentration (ng/μL)',
    'Phenylacetaldehyde (μM)', 'Styrene oxide (μM)',
    'All styrene products (μM)', 'CPE_ll', 'Prior_ll', 'Finetuned_ll',
    'Vdist_ll', 'Acq', 'Rank'
]
for c in corr_cols:
    df_summary[c] = pd.to_numeric(df_summary[c], errors="coerce")
corr = df_summary[corr_cols].corr(method="spearman")
corr.style.background_gradient(cmap='coolwarm').format(precision=2)

In [ ]:
corr.to_csv(RESPATH / "validation_spearman_corr.csv", encoding="utf-8-sig")

## Generate novel sequences

Get likelihoods, and statics on residues mutated (sites).

In [ ]:
Xnov, logP_Xnov, acq_Xnov, cpe_Xnov = [], [], [], []
for _ in range(10000 // VSD_BATCH_SIZE):
    x, lp = gen(samples=VSD_BATCH_SIZE)
    acq = vsacq(x, lp).detach().cpu().numpy()
    lcpe = cpe(x).detach().cpu().numpy()

    Xnov.append(x.detach().cpu())
    logP_Xnov.append(lp.detach().cpu().numpy())
    acq_Xnov.append(acq)
    cpe_Xnov.append(lcpe)

Xnov = torch.vstack(Xnov)
logP_Xnov = np.concatenate(logP_Xnov)
acq_Xnov = np.concatenate(acq_Xnov)
cpe_Xnov = np.concatenate(cpe_Xnov)

In [ ]:
diff = (Xnov - wt_tokens) > 0
nmutcounts = diff.sum(axis=0)
f, axs = plt.subplots(nrows=1, ncols=1, figsize=(17, 5), dpi=150)

axs.bar(np.arange(len(WTSEQ)), nmutcounts, label="all sites")
axs.bar(MUT_SITES, nmutcounts[MUT_SITES], color="r", label="targeted")
axs.legend()
axs.grid()
axs.set_title("Posterior Sampled sequences")
plt.show()


In [ ]:
# Drop variants with mutations outside of the 10 sites
not_mut_sites = list(set(np.arange(len(WTSEQ))).difference(MUT_SITES))
mut_at_sites = diff[:, not_mut_sites].sum(axis=1) == 0
Xnov = Xnov[mut_at_sites, :]
logP_Xnov = logP_Xnov[mut_at_sites]
cpe_Xnov = cpe_Xnov[mut_at_sites]
acq_Xnov = acq_Xnov[mut_at_sites]

In [ ]:
seq_nov = intarr2seq(Xnov)
df_nov = pd.DataFrame(dict(
    Variants=seq_nov,
    Vdist_ll=logP_Xnov,
    CPE_ll=cpe_Xnov,
    Acq=acq_Xnov,
))
df_nov = df_nov.drop_duplicates(subset="Variants")

# Remove variants not starting with methionine
valid = np.array([s[0] == "M" for s in df_nov["Variants"]])
df_nov = df_nov[valid]

# Drop variants that are in the validation data
df_nov = df_nov.set_index("Variants")
val_seqs = df_val.loc[~df_val["Phenylacetaldehyde (μM)"].isna(), "Variants"]
df_nov = df_nov.drop(index=val_seqs, errors="ignore")

In [ ]:
df_nov[["Vdist_ll", "Acq", "CPE_ll"]].describe()

In [ ]:
df_nov.sort_values(by=["Vdist_ll"], ascending=False)

In [ ]:
df_nov.sort_values(by=["Acq"], ascending=False)

### Generated sequence checks

In [ ]:
# Proportion of generated sequences in training data

n_both = len(set(S).intersection(seq_nov))
print(f"Generated sequences in training data: # {n_both}, proportion {n_both / len(df_nov):.3f}")

In [ ]:
# Diversity of training vs generated sequences
div_train = wt_diversity(S)
div_nov = wt_diversity(df_nov.index)

print(f"Diversity: train = {div_train:.3f}, gen = {div_nov:.3f}.")

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
df_nov.plot.scatter(x="CPE_ll", y="Vdist_ll", ax=axs[0], alpha=0.1)
df_nov.plot.scatter(x="CPE_ll", y="Acq", ax=axs[1], alpha=0.1)
df_nov.plot.scatter(x="Vdist_ll", y="Acq", ax=axs[2], alpha=0.1)
plt.show()

## Save generated sequences and scores 

In [ ]:
df_nov.to_csv(RESPATH / "generated_upo.csv", encoding="utf-8-sig")

In [ ]:
corr_cols = ["Vdist_ll", "CPE_ll", "Acq"]
corr = df_nov[corr_cols].corr(method="spearman")
corr.style.background_gradient(cmap='coolwarm').format(precision=2)